In [ ]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
# ── 0. Install ────────────────────────────────────────────────────────────────
import subprocess, sys
for pkg in ['catboost', 'lightgbm', 'xgboost', 'optuna']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

# ── 1. Imports ────────────────────────────────────────────────────────────────
import warnings, gc, os
warnings.filterwarnings('ignore')

import numpy  as np
import pandas as pd
from pathlib import Path

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection  import StratifiedKFold
from sklearn.preprocessing    import LabelEncoder
from sklearn.impute           import SimpleImputer
from sklearn.metrics          import accuracy_score
from sklearn.linear_model     import LogisticRegression
from sklearn.ensemble         import ExtraTreesClassifier, RandomForestClassifier

import lightgbm as lgb
import xgboost  as xgb
from catboost import CatBoostClassifier

SEED = 42
np.random.seed(SEED)

# ── 2. Load Data ──────────────────────────────────────────────────────────────
DATA      = Path('/kaggle/input/competitions/spaceship-titanic')
train_raw = pd.read_csv(DATA / 'train.csv')
test_raw  = pd.read_csv(DATA / 'test.csv')
sub       = pd.read_csv(DATA / 'sample_submission.csv')
print(f'Train: {train_raw.shape} | Test: {test_raw.shape}')

SPEND_COLS = ['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']
TARGET     = 'Transported'

# ── 3. Feature Engineering ────────────────────────────────────────────────────
def engineer(df):
    df = df.copy()

    # PassengerId
    df['GroupID']      = df['PassengerId'].str.split('_').str[0]
    df['PersonNum']    = df['PassengerId'].str.split('_').str[1].astype(int)
    df['GroupIDNum']   = df['GroupID'].astype(int)
    grp_size           = df.groupby('GroupID')['PassengerId'].transform('count')
    df['GroupSize']    = grp_size
    df['IsSolo']       = (grp_size == 1).astype(int)
    df['IsLargeGroup'] = (grp_size >= 4).astype(int)

    # Name → LastName (safe feature — used only for TE, not label prop)
    df['LastName'] = df['Name'].str.split(' ').str[-1].fillna('Unknown')

    # Cabin
    cabin          = df['Cabin'].str.split('/', expand=True)
    df['Deck']     = cabin[0]
    df['CabinNum'] = pd.to_numeric(cabin[1], errors='coerce')
    df['Side']     = cabin[2]

    # Spend
    for c in SPEND_COLS:
        df[c] = df[c].fillna(0)
    df['TotalSpend']  = df[SPEND_COLS].sum(axis=1)
    df['HasSpend']    = (df['TotalSpend'] > 0).astype(int)
    df['SpendLog']    = np.log1p(df['TotalSpend'])
    df['NumServices'] = (df[SPEND_COLS] > 0).sum(axis=1)
    df['SpendMax']    = df[SPEND_COLS].max(axis=1)
    df['SpendVar']    = df[SPEND_COLS].var(axis=1).fillna(0)
    df['SpendSkew']   = df[SPEND_COLS].skew(axis=1).fillna(0)
    for c in SPEND_COLS:
        df[f'{c}Log']   = np.log1p(df[c])
        df[f'{c}Ratio'] = df[c] / (df['TotalSpend'] + 1e-9)

    # CryoSleep ↔ spend imputation
    df.loc[df['TotalSpend'] > 0,  'CryoSleep'] = df.loc[df['TotalSpend'] > 0,  'CryoSleep'].fillna(False)
    df.loc[df['TotalSpend'] == 0, 'CryoSleep'] = df.loc[df['TotalSpend'] == 0, 'CryoSleep'].fillna(True)
    cryo = df['CryoSleep'] == True
    for c in SPEND_COLS:
        df.loc[cryo, c] = 0

    # Group-level HomePlanet / Destination imputation
    for col in ['HomePlanet', 'Destination']:
        grp_mode = df.groupby('GroupID')[col].transform(
            lambda s: s.mode()[0] if s.notna().any() else np.nan)
        df[col] = df[col].fillna(grp_mode)

    deck_planet = {'A':'Europa','B':'Europa','C':'Europa','T':'Europa','G':'Earth'}
    df['HomePlanet'] = df.apply(
        lambda r: deck_planet.get(r['Deck'], r['HomePlanet'])
        if pd.isna(r['HomePlanet']) else r['HomePlanet'], axis=1)

    # Age: impute by Deck+Side median (smarter than global)
    df['Age'] = df['Age'].fillna(
        df.groupby(['Deck','Side'])['Age'].transform('median')).fillna(
        df.groupby('Deck')['Age'].transform('median')).fillna(df['Age'].median())

    # Group spend aggregates
    for fn, nm in [('mean','Mean'),('std','Std'),('max','Max'),('min','Min'),('median','Med')]:
        df[f'GrpSpend{nm}'] = df.groupby('GroupID')['TotalSpend'].transform(fn).fillna(0)
    df['GrpSpendStd'] = df['GrpSpendStd'].fillna(0)
    df['GrpSpendRel'] = df['TotalSpend'] / (df['GrpSpendMean'] + 1)  # spend vs group mean

    # Group CryoSleep proportion
    cryo_int = df['CryoSleep'].map({True:1,False:0,'True':1,'False':0}).fillna(0.5)
    tmp = cryo_int.copy(); tmp.index = df.index
    grp_cryo = df.groupby('GroupID').apply(lambda g: tmp.loc[g.index].mean())
    df['GrpCryoProp'] = df['GroupID'].map(grp_cryo)

    # Age bins
    df['AgeBin'] = pd.cut(df['Age'],
        bins=[-1,5,12,17,25,35,50,65,200],
        labels=['Baby','Child','Teen','YoungAdult','Adult','MidAge','Senior','Elder'])
    df['IsChild']  = (df['Age'] < 13).astype(float)
    df['AgeGroup'] = df['Age'] // 10

    # Interactions
    df['DeckSide']     = df['Deck'].astype(str) + '_' + df['Side'].astype(str)
    df['DeckDest']     = df['Deck'].astype(str) + '_' + df['Destination'].astype(str)
    df['PlanetDeck']   = df['HomePlanet'].astype(str) + '_' + df['Deck'].astype(str)
    df['PlanetDest']   = df['HomePlanet'].astype(str) + '_' + df['Destination'].astype(str)
    df['DeckSideDest'] = df['DeckSide'] + '_' + df['Destination'].astype(str)
    df['CabinParity']  = (df['CabinNum'] % 2).fillna(-1).astype(int)
    df['CabinRegion']  = pd.qcut(
        df['CabinNum'].fillna(df['CabinNum'].median()),
        q=6, labels=False, duplicates='drop')
    df['CabinBucket']  = (df['Deck'].astype(str) + '_' +
        (df['CabinNum'].fillna(-1) // 50).astype(int).astype(str))

    # Bool → int
    df['CryoSleep_int']   = df['CryoSleep'].map({True:1,False:0,'True':1,'False':0}).fillna(0.5)
    df['VIP_int']         = df['VIP'].map({True:1,False:0,'True':1,'False':0}).fillna(0)
    df['Cryo_x_VIP']      = df['CryoSleep_int'] * df['VIP_int']
    df['Cryo_x_Spend']    = df['CryoSleep_int'] * df['SpendLog']
    df['NotCryo_x_Spend'] = (1 - df['CryoSleep_int']) * df['SpendLog']

    # Extra interactions
    df['SpendPerAge']     = df['TotalSpend'] / (df['Age'] + 1)
    df['AgeSpendBin']     = (df['AgeBin'].astype(str) + '_' +
                              pd.cut(df['TotalSpend'], bins=[-1,0,100,1000,100000],
                              labels=['zero','low','mid','high']).astype(str))
    df['DeckCryo']        = df['Deck'].astype(str) + '_' + df['CryoSleep_int'].astype(str)
    df['PlanetCryo']      = df['HomePlanet'].astype(str) + '_' + df['CryoSleep_int'].astype(str)

    return df

train_fe = engineer(train_raw)
test_fe  = engineer(test_raw)
print(f'After FE — train: {train_fe.shape}, test: {test_fe.shape}')

# ── 4. CV-Safe Target Encoding ────────────────────────────────────────────────
SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

TE_COLS = ['Deck','Side','HomePlanet','Destination',
           'DeckSide','DeckDest','PlanetDeck','PlanetDest',
           'DeckSideDest','AgeBin','AgeGroup','CabinBucket',
           'LastName','DeckCryo','PlanetCryo','AgeSpendBin']

SMOOTH      = 30
global_mean = train_fe[TARGET].mean()
y_tr        = train_fe[TARGET].values

te_train_dict = {c: np.zeros(len(train_fe)) for c in TE_COLS}
te_test_dict  = {c: np.zeros(len(test_fe))  for c in TE_COLS}

for fold, (tr_idx, vl_idx) in enumerate(SKF.split(train_fe, y_tr)):
    fold_tr = train_fe.iloc[tr_idx]
    fold_vl = train_fe.iloc[vl_idx]
    for c in TE_COLS:
        stats       = fold_tr.groupby(c)[TARGET].agg(['sum','count'])
        stats['te'] = (stats['sum'] + SMOOTH*global_mean) / (stats['count'] + SMOOTH)
        te_map      = stats['te'].to_dict()
        te_train_dict[c][vl_idx] = fold_vl[c].astype(str).map(te_map).fillna(global_mean).values
        full_stats       = train_fe.groupby(c)[TARGET].agg(['sum','count'])
        full_stats['te'] = (full_stats['sum'] + SMOOTH*global_mean) / (full_stats['count'] + SMOOTH)
        te_test_dict[c] += test_fe[c].astype(str).map(full_stats['te'].to_dict()).fillna(global_mean).values / 5

for c in TE_COLS:
    train_fe[f'TE_{c}'] = te_train_dict[c]
    test_fe [f'TE_{c}'] = te_test_dict[c]
print('Target encoding done')

# ── 5. Feature Matrix ─────────────────────────────────────────────────────────
CAT_COLS = ['HomePlanet','Destination','Deck','Side',
            'DeckSide','DeckDest','PlanetDeck','PlanetDest','DeckSideDest',
            'AgeBin','GroupID','CabinBucket','LastName','DeckCryo','PlanetCryo']

NUM_COLS = (['Age','RoomService','FoodCourt','ShoppingMall','Spa','VRDeck',
             'TotalSpend','SpendLog','SpendMax','SpendVar','SpendSkew','NumServices',
             'GroupSize','GroupIDNum','PersonNum','IsSolo','IsLargeGroup',
             'CabinNum','CabinRegion','CabinParity',
             'GrpSpendMean','GrpSpendStd','GrpSpendMax','GrpSpendMin','GrpSpendMed','GrpSpendRel',
             'GrpCryoProp',
             'CryoSleep_int','VIP_int','Cryo_x_VIP','Cryo_x_Spend','NotCryo_x_Spend',
             'IsChild','AgeGroup','HasSpend','SpendPerAge'] +
            [f'{c}Log'   for c in SPEND_COLS] +
            [f'{c}Ratio' for c in SPEND_COLS] +
            [f'TE_{c}'   for c in TE_COLS])

for c in CAT_COLS:
    le = LabelEncoder()
    combined = pd.concat([train_fe[c], test_fe[c]], axis=0).astype(str).fillna('nan')
    le.fit(combined)
    train_fe[c] = le.transform(train_fe[c].astype(str).fillna('nan'))
    test_fe [c] = le.transform(test_fe [c].astype(str).fillna('nan'))

ALL_FEATS = NUM_COLS + CAT_COLS
X_all     = pd.concat([train_fe[ALL_FEATS], test_fe[ALL_FEATS]], axis=0)
imp       = SimpleImputer(strategy='median')
X_all_imp = pd.DataFrame(imp.fit_transform(X_all), columns=ALL_FEATS)

n_tr    = len(train_fe)
X_train = X_all_imp.iloc[:n_tr].reset_index(drop=True)
X_test  = X_all_imp.iloc[n_tr:].reset_index(drop=True)
y_train = train_fe[TARGET].astype(int).reset_index(drop=True)
y_np    = y_train.values
print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')

Xaug      = X_train.values.astype(np.float32)
Xaug_test = X_test.values.astype(np.float32)

# ── 6. Optuna Tuning ──────────────────────────────────────────────────────────
N_TRIALS = 40

def obj_cb(trial):
    p = dict(
        iterations=trial.suggest_int('iters',600,2500),
        learning_rate=trial.suggest_float('lr',0.01,0.15,log=True),
        depth=trial.suggest_int('depth',4,10),
        l2_leaf_reg=trial.suggest_float('l2',1,15),
        random_strength=trial.suggest_float('rs',0.1,3.0),
        bagging_temperature=trial.suggest_float('bt',0.0,1.5),
        min_data_in_leaf=trial.suggest_int('mdil',1,40),
        random_seed=SEED, verbose=0, loss_function='Logloss', eval_metric='Accuracy')
    scores = []
    for ti, vi in SKF.split(Xaug, y_np):
        m = CatBoostClassifier(**p)
        m.fit(Xaug[ti], y_np[ti], eval_set=(Xaug[vi], y_np[vi]),
              early_stopping_rounds=60, verbose=0)
        scores.append(accuracy_score(y_np[vi], m.predict(Xaug[vi])))
    return np.mean(scores)

def obj_lgb(trial):
    p = dict(
        n_estimators=trial.suggest_int('ne',500,3000),
        learning_rate=trial.suggest_float('lr',0.005,0.15,log=True),
        max_depth=trial.suggest_int('md',3,10),
        num_leaves=trial.suggest_int('nl',20,200),
        min_child_samples=trial.suggest_int('mcs',5,100),
        subsample=trial.suggest_float('ss',0.5,1.0),
        colsample_bytree=trial.suggest_float('cst',0.4,1.0),
        reg_alpha=trial.suggest_float('ra',0.0,3.0),
        reg_lambda=trial.suggest_float('rl',0.0,3.0),
        subsample_freq=1,
        random_state=SEED, verbose=-1, n_jobs=-1)
    scores = []
    for ti, vi in SKF.split(Xaug, y_np):
        m = lgb.LGBMClassifier(**p)
        m.fit(Xaug[ti], y_np[ti], eval_set=[(Xaug[vi], y_np[vi])],
              callbacks=[lgb.early_stopping(60,verbose=False), lgb.log_evaluation(-1)])
        scores.append(accuracy_score(y_np[vi], m.predict(Xaug[vi])))
    return np.mean(scores)

def obj_xgb(trial):
    p = dict(
        n_estimators=trial.suggest_int('ne',500,3000),
        learning_rate=trial.suggest_float('lr',0.005,0.15,log=True),
        max_depth=trial.suggest_int('md',3,10),
        subsample=trial.suggest_float('ss',0.5,1.0),
        colsample_bytree=trial.suggest_float('cst',0.4,1.0),
        colsample_bylevel=trial.suggest_float('cbl',0.4,1.0),
        reg_alpha=trial.suggest_float('ra',0.0,3.0),
        reg_lambda=trial.suggest_float('rl',0.0,6.0),
        min_child_weight=trial.suggest_int('mcw',1,15),
        gamma=trial.suggest_float('gm',0.0,2.0),
        early_stopping_rounds=60,
        random_state=SEED, eval_metric='logloss',
        n_jobs=-1, verbosity=0)
    scores = []
    for ti, vi in SKF.split(Xaug, y_np):
        m = xgb.XGBClassifier(**p)
        m.fit(Xaug[ti], y_np[ti], eval_set=[(Xaug[vi], y_np[vi])], verbose=False)
        scores.append(accuracy_score(y_np[vi], m.predict(Xaug[vi])))
    return np.mean(scores)

print('Tuning CatBoost...')
s_cb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
s_cb.optimize(obj_cb, n_trials=N_TRIALS)
print(f'  CB best: {s_cb.best_value:.5f}')

print('Tuning LightGBM...')
s_lgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
s_lgb.optimize(obj_lgb, n_trials=N_TRIALS)
print(f'  LGB best: {s_lgb.best_value:.5f}')

print('Tuning XGBoost...')
s_xgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
s_xgb.optimize(obj_xgb, n_trials=N_TRIALS)
print(f'  XGB best: {s_xgb.best_value:.5f}')

# ── 7. Rename best params ─────────────────────────────────────────────────────
def rk(d, m): return {m.get(k, k): v for k, v in d.items()}

CB_P  = rk(s_cb.best_params,  {'lr':'learning_rate','l2':'l2_leaf_reg',
    'rs':'random_strength','bt':'bagging_temperature',
    'mdil':'min_data_in_leaf','iters':'iterations'})
LGB_P = rk(s_lgb.best_params, {'lr':'learning_rate','ne':'n_estimators','md':'max_depth',
    'nl':'num_leaves','mcs':'min_child_samples','ss':'subsample',
    'cst':'colsample_bytree','ra':'reg_alpha','rl':'reg_lambda'})
XGB_P = rk(s_xgb.best_params, {'lr':'learning_rate','ne':'n_estimators','md':'max_depth',
    'ss':'subsample','cst':'colsample_bytree','cbl':'colsample_bylevel',
    'ra':'reg_alpha','rl':'reg_lambda','mcw':'min_child_weight','gm':'gamma'})
XGB_P['early_stopping_rounds'] = 60

# ── 8. Level-1 OOF ────────────────────────────────────────────────────────────
oof_cb  = np.zeros(n_tr); tst_cb  = np.zeros(len(X_test))
oof_lgb = np.zeros(n_tr); tst_lgb = np.zeros(len(X_test))
oof_xgb = np.zeros(n_tr); tst_xgb = np.zeros(len(X_test))
oof_et  = np.zeros(n_tr); tst_et  = np.zeros(len(X_test))
oof_rf  = np.zeros(n_tr); tst_rf  = np.zeros(len(X_test))

print('Running Level-1 OOF...')
for fold, (ti, vi) in enumerate(SKF.split(Xaug, y_np)):
    Xtr_, Xvl_ = Xaug[ti], Xaug[vi]
    ytr_, yvl_ = y_np[ti], y_np[vi]

    cb_m = CatBoostClassifier(**CB_P, random_seed=SEED, verbose=0,
                               loss_function='Logloss', eval_metric='Accuracy')
    cb_m.fit(Xtr_, ytr_, eval_set=(Xvl_, yvl_), early_stopping_rounds=60, verbose=0)
    oof_cb[vi]  = cb_m.predict_proba(Xvl_)[:,1]
    tst_cb     += cb_m.predict_proba(Xaug_test)[:,1] / 5

    lgb_m = lgb.LGBMClassifier(**LGB_P, random_state=SEED, verbose=-1, n_jobs=-1)
    lgb_m.fit(Xtr_, ytr_, eval_set=[(Xvl_, yvl_)],
              callbacks=[lgb.early_stopping(60,verbose=False), lgb.log_evaluation(-1)])
    oof_lgb[vi] = lgb_m.predict_proba(Xvl_)[:,1]
    tst_lgb    += lgb_m.predict_proba(Xaug_test)[:,1] / 5

    xgb_m = xgb.XGBClassifier(**XGB_P, random_state=SEED, verbosity=0,
                                eval_metric='logloss', n_jobs=-1)
    xgb_m.fit(Xtr_, ytr_, eval_set=[(Xvl_, yvl_)], verbose=False)
    oof_xgb[vi] = xgb_m.predict_proba(Xvl_)[:,1]
    tst_xgb    += xgb_m.predict_proba(Xaug_test)[:,1] / 5

    et_m = ExtraTreesClassifier(n_estimators=600, min_samples_leaf=2,
                                 max_features=0.7, n_jobs=-1, random_state=SEED)
    et_m.fit(Xtr_, ytr_)
    oof_et[vi]  = et_m.predict_proba(Xvl_)[:,1]
    tst_et     += et_m.predict_proba(Xaug_test)[:,1] / 5

    rf_m = RandomForestClassifier(n_estimators=500, min_samples_leaf=2,
                                   max_features=0.7, n_jobs=-1, random_state=SEED+fold)
    rf_m.fit(Xtr_, ytr_)
    oof_rf[vi]  = rf_m.predict_proba(Xvl_)[:,1]
    tst_rf     += rf_m.predict_proba(Xaug_test)[:,1] / 5

    print(f'  Fold {fold+1} | CB={accuracy_score(yvl_,oof_cb[vi]>0.5):.5f} '
          f'LGB={accuracy_score(yvl_,oof_lgb[vi]>0.5):.5f} '
          f'XGB={accuracy_score(yvl_,oof_xgb[vi]>0.5):.5f} '
          f'ET={accuracy_score(yvl_,oof_et[vi]>0.5):.5f} '
          f'RF={accuracy_score(yvl_,oof_rf[vi]>0.5):.5f}')

print(f'\nL1 OOF | CB={accuracy_score(y_np,oof_cb>0.5):.5f} '
      f'LGB={accuracy_score(y_np,oof_lgb>0.5):.5f} '
      f'XGB={accuracy_score(y_np,oof_xgb>0.5):.5f} '
      f'ET={accuracy_score(y_np,oof_et>0.5):.5f} '
      f'RF={accuracy_score(y_np,oof_rf>0.5):.5f}')

# ── 9. Pseudo-labeling ────────────────────────────────────────────────────────
avg_test    = (tst_cb + tst_lgb + tst_xgb + tst_et + tst_rf) / 5
pseudo_mask = (avg_test > 0.92) | (avg_test < 0.08)
n_pseudo    = pseudo_mask.sum()
print(f'Pseudo-labels: {n_pseudo} / {len(pseudo_mask)}')

if n_pseudo > 50:
    X_pseudo = Xaug_test[pseudo_mask]
    y_pseudo = (avg_test[pseudo_mask] > 0.5).astype(int)
    oof_cb2  = np.zeros(n_tr); tst_cb2  = np.zeros(len(X_test))
    oof_lgb2 = np.zeros(n_tr); tst_lgb2 = np.zeros(len(X_test))
    for fold, (ti, vi) in enumerate(SKF.split(Xaug, y_np)):
        Xtr_pl = np.concatenate([Xaug[ti], X_pseudo])
        ytr_pl = np.concatenate([y_np[ti], y_pseudo])
        Xvl_, yvl_ = Xaug[vi], y_np[vi]
        cb2 = CatBoostClassifier(**CB_P, random_seed=SEED, verbose=0,
                                  loss_function='Logloss', eval_metric='Accuracy')
        cb2.fit(Xtr_pl, ytr_pl, eval_set=(Xvl_, yvl_), early_stopping_rounds=60, verbose=0)
        oof_cb2[vi] = cb2.predict_proba(Xvl_)[:,1]
        tst_cb2    += cb2.predict_proba(Xaug_test)[:,1] / 5
        lgb2 = lgb.LGBMClassifier(**LGB_P, random_state=SEED, verbose=-1, n_jobs=-1)
        lgb2.fit(Xtr_pl, ytr_pl, eval_set=[(Xvl_, yvl_)],
                 callbacks=[lgb.early_stopping(60,verbose=False), lgb.log_evaluation(-1)])
        oof_lgb2[vi] = lgb2.predict_proba(Xvl_)[:,1]
        tst_lgb2    += lgb2.predict_proba(Xaug_test)[:,1] / 5
    if accuracy_score(y_np, oof_cb2>0.5) > accuracy_score(y_np, oof_cb>0.5):
        oof_cb, tst_cb = oof_cb2, tst_cb2
        print('  -> CB improved with pseudo-labels')
    if accuracy_score(y_np, oof_lgb2>0.5) > accuracy_score(y_np, oof_lgb>0.5):
        oof_lgb, tst_lgb = oof_lgb2, tst_lgb2
        print('  -> LGB improved with pseudo-labels')

# ── 10. Level-2 Meta + Level-3 Blender ───────────────────────────────────────
L1_tr = np.column_stack([oof_cb, oof_lgb, oof_xgb, oof_et, oof_rf,
    oof_cb*oof_lgb, oof_cb*oof_xgb, oof_lgb*oof_xgb, oof_lgb*oof_rf,
    (oof_cb+oof_lgb+oof_xgb)/3,
    (oof_cb+oof_lgb+oof_xgb+oof_et+oof_rf)/5])
L1_te = np.column_stack([tst_cb, tst_lgb, tst_xgb, tst_et, tst_rf,
    tst_cb*tst_lgb, tst_cb*tst_xgb, tst_lgb*tst_xgb, tst_lgb*tst_rf,
    (tst_cb+tst_lgb+tst_xgb)/3,
    (tst_cb+tst_lgb+tst_xgb+tst_et+tst_rf)/5])

print('Level-2: Meta-LightGBM...')
oof_meta = np.zeros(n_tr)
tst_meta = np.zeros(len(X_test))
for fold, (ti, vi) in enumerate(SKF.split(L1_tr, y_np)):
    meta_m = lgb.LGBMClassifier(
        n_estimators=600, learning_rate=0.01, max_depth=3, num_leaves=15,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1,
        random_state=SEED, verbose=-1, n_jobs=-1)
    meta_m.fit(L1_tr[ti], y_np[ti], eval_set=[(L1_tr[vi], y_np[vi])],
               callbacks=[lgb.early_stopping(40,verbose=False), lgb.log_evaluation(-1)])
    oof_meta[vi] = meta_m.predict_proba(L1_tr[vi])[:,1]
    tst_meta    += meta_m.predict_proba(L1_te)[:,1] / 5
print(f'  L2 OOF: {accuracy_score(y_np, oof_meta>0.5):.5f}')

print('Level-3: LR Blender...')
L2_tr = np.column_stack([oof_meta, oof_cb, oof_lgb, oof_xgb, oof_et, oof_rf])
L2_te = np.column_stack([tst_meta, tst_cb, tst_lgb, tst_xgb, tst_et, tst_rf])
oof_blend = np.zeros(n_tr)
tst_blend = np.zeros(len(X_test))
for ti, vi in SKF.split(L2_tr, y_np):
    lr_m = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
    lr_m.fit(L2_tr[ti], y_np[ti])
    oof_blend[vi] = lr_m.predict_proba(L2_tr[vi])[:,1]
    tst_blend    += lr_m.predict_proba(L2_te)[:,1] / 5
print(f'  L3 Blend OOF: {accuracy_score(y_np, oof_blend>0.5):.5f}')

# ── 11. Summary + Threshold + Save ───────────────────────────────────────────
print('='*60)
for name, oof in [('CB',oof_cb),('LGB',oof_lgb),('XGB',oof_xgb),
                   ('ET',oof_et),('RF',oof_rf),('Meta-L2',oof_meta),('Blend-L3',oof_blend)]:
    print(f'  {name:<10}: {accuracy_score(y_np, oof>0.5):.5f}')
print('='*60)

best_thr, best_acc = 0.5, 0.0
for thr in np.arange(0.35, 0.65, 0.005):
    a = accuracy_score(y_np, oof_blend > thr)
    if a > best_acc:
        best_acc, best_thr = a, thr
print(f'Optimal threshold: {best_thr:.3f}  OOF acc: {best_acc:.5f}')

sub['Transported'] = (tst_blend > best_thr)
out_path = '/kaggle/working/submission.csv'
sub.to_csv(out_path, index=False)

if os.path.exists(out_path):
    print(f'✓ Saved: {out_path} ({os.path.getsize(out_path)} bytes)')
else:
    sub.to_csv('submission.csv', index=False)
    print('✓ Saved to cwd fallback')

print(sub['Transported'].value_counts())
sub.head()

In [ ]:
import os

print("Current files:")
print(os.listdir("/kaggle/working"))